## Executive Summary

**Purpose:** Explain what changes when a single gear stage uses helical teeth instead of spur teeth, and how the `emachines.gears.helical` notebooks take a requirement to a manufacturing package.

**What it covers:**

1. What a helical tooth is, and why it runs quieter and carries more than a spur tooth of the same size.
2. Normal and transverse sections, helix angle, hand, lead and overlap, with a drawing of the pitch cylinder unrolled.
3. The axial force, and what it means for bearings and shaft joints.
4. The workflow and the data that define a design.

## How It Fits Into the Helical Gear Workflow

```
00_overview → 01_geometry → 02_layout → 03_rating → 04_components → 05_manufacturing → 06_design → 07_api
 (this page)   normal and    tooth        ISO 6336     hubs, shafts,     tolerances,         search,     public
               transverse,   counts,      with helix   bearings with     backlash, data      report,     imports
               ε_α, ε_β,     helix angle  factors,     axial force       tables, DXF         package
               W_k, M_d      by priority  peak check
```

The helical module reuses the spur tools of `emachines.gears` wherever the physics is the same. Every transverse section of a helical gear is a spur gear (the *transverse equivalent*). The tooth strength is that of a spur gear with more teeth (the *virtual gear*). Materials, life factors, fits, keys and press fits are shared.

In [ ]:
#| hide
import math
import numpy as np
import matplotlib.pyplot as plt
from emachines.gears.helical.layout import HelicalPair

## 1. The Helical Tooth

On a helical gear the teeth are not parallel to the axis. They wind around the gear like a steep screw thread, at the **helix angle** $\beta$ on the reference cylinder. Two external helical gears mesh on parallel shafts when they have the same helix angle with **opposite hands**. A right-hand gear meshes with a left-hand one.

Three things follow from the twist:

- **Contact starts gradually.** A spur tooth comes into mesh along its whole face width at once. A helical tooth comes into mesh at one end and the contact line sweeps across the face. The mesh stiffness changes smoothly, so the gear runs quieter and excites the housing less. That is the main reason for helical gears in motor-driven stages above a few hundred rpm.
- **More teeth share the load.** Besides the transverse contact ratio $\varepsilon_\alpha$ of the spur equivalent, each tooth stays in contact for $\varepsilon_\beta = b\sin\beta/(\pi m_n)$ more pitches across the face. So $\varepsilon_\gamma = \varepsilon_\alpha + \varepsilon_\beta$ teeth carry the load on average, typically 2.5–3.5 against 1.4–1.7 for spur gears.
- **The tooth force has an axial component.** The force acts normal to the tooth flank. That normal is tilted by the base helix angle, so part of the force pushes along the shaft: $F_a = F_t\tan\beta_w$. The bearings must take it, and the gear must be held on its shaft against it.

## 2. Normal and Transverse Sections

Unroll the pitch cylinder and the teeth become straight lines at the angle $\beta$ to the axis. Two sections matter:

- the **normal section**, perpendicular to the teeth. This is where the hob cuts, so the normal module $m_n$ and normal pressure angle $\alpha_n$ are the standard tool values.
- the **transverse section**, perpendicular to the axis. This is where the gear works as a spur gear, with module $m_t = m_n/\cos\beta$ and pressure angle $\tan\alpha_t = \tan\alpha_n/\cos\beta$.

The **axial pitch** $p_x = \pi m_n/\sin\beta$ is the axial distance between neighbouring teeth. The **lead** $p_z = \pi d/\tan\beta$ is the axial advance of one tooth over a full turn, the figure the gear shop needs to set up the machine.

In [ ]:
#| echo: false
def _unrolled():
    pair = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(20), b=20)
    g = pair.pinion
    fig, ax = plt.subplots(figsize=(11, 4.4))
    pt, b, beta = g.pt, g.b, g.beta
    W = 3.2 * pt
    # teeth as bands (unrolled reference cylinder: x along circumference, y along axis)
    for k in range(-2, 6):
        x0 = k * pt
        s = g.st
        xs = np.array([x0, x0 + s, x0 + s + b * math.tan(beta), x0 + b * math.tan(beta)])
        ys = np.array([0, 0, b, b])
        ax.fill(xs, ys, fc="#d8e8d0", ec="#222", lw=0.7)
    ax.add_patch(plt.Rectangle((0, 0), W, b, fill=False, ec="#888", lw=0.8, ls="--"))
    ax.set_xlim(-0.2 * pt, W + 0.2 * pt); ax.set_ylim(-6, b + 6)
    # pitches
    y0 = -2.5
    ax.annotate("", xy=(pt, y0), xytext=(0, y0), arrowprops=dict(arrowstyle="<->", color="#1f5fa8"))
    ax.text(pt / 2, y0 - 1.6, "p_t (transverse)", color="#1f5fa8", ha="center", fontsize=9)
    xn = 2 * pt
    nx, ny = math.cos(beta), -math.sin(beta)
    ax.annotate("", xy=(xn + g.pn * nx, b / 2 + g.pn * ny), xytext=(xn, b / 2),
                arrowprops=dict(arrowstyle="<->", color="#b8322a"))
    ax.text(xn + g.pn * nx / 2 + 0.6, b / 2 + g.pn * ny / 2 - 1.2, "p_n (normal)", color="#b8322a", fontsize=9)
    ax.plot([0, 0], [0, b], color="#2a8a4a", lw=0.8)
    ax.plot([0, b * math.tan(beta)], [0, b], color="#2a8a4a", lw=0.8)
    ax.text(0.8, b * 0.55, "β", color="#2a8a4a", fontsize=11)
    ax.annotate("", xy=(W + 0.1 * pt, b), xytext=(W + 0.1 * pt, 0), arrowprops=dict(arrowstyle="<->", color="#444"))
    ax.text(W + 0.15 * pt, b / 2, "b", fontsize=10, va="center")
    ax.set_xlabel("circumference on the reference cylinder [mm]"); ax.set_ylabel("axial position [mm]")
    ax.set_title(f"Reference cylinder unrolled: z = {g.z}, m_n = {g.mn:g}, β = 20°, b = {b:g} mm, "
                 f"p_x = {g.px:.1f} mm, ε_β = {pair.mesh.eps_beta:.2f}", fontsize=10)
    ax.set_aspect("equal")
    fig.tight_layout(); plt.show()
_unrolled()

## 3. Axial Force, Bearings and Joints

With pinion torque $T_1$ and operating pitch diameter $d_{w1}$, the mesh force splits into

| Component | Value | Taken by |
|:--|:--|:--|
| tangential $F_t$ | $2000\,T_1/d_{w1}$ | torque. It bends the shafts in the tangential plane |
| radial $F_r$ | $F_t\tan\alpha_{wt}$ | pushes the gears apart. It bends the shafts in the plane of the axes |
| axial $F_a$ | $F_t\tan\beta_w$ | the **located** bearing of each shaft. It also tilts the shaft through the moment $F_a r_w$ |

The axial force reverses with the torque. A motor drive that runs both ways or brakes must locate each shaft in both directions. A key does not carry it, so the gear needs a shoulder and a retaining ring or nut. A press fit carries it by friction together with the torque. The **helix angle** is therefore a judgement between smoothness (high $\beta$, high $\varepsilon_\beta$) and thrust (low $\beta$). `gears.helical.layout` offers three priorities, `'quiet'`, `'balanced'` and `'low_thrust'`, and the design report says which angle was chosen and why.

## 4. What Defines a Design

| Group | Data | Where |
|:--|:--|:--|
| Requirement | ratio, rated torque and speed, life, peak torque, shaft diameters, fixed centre distance (optional), priority | `06_design.HelicalRequirements` |
| Gear pair | $z_1$, $z_2$, $m_n$, $\alpha_n$, $\beta$, hands, $x_1$, $x_2$, $k^*$, $b$, $a_w$ | `02_layout.HelicalPair` |
| Checks | $\varepsilon_\alpha$, $\varepsilon_\beta$, $\varepsilon_\gamma$, undercut, tip thickness, interference, clearance | `01_geometry.helical_mesh_check` |
| Strength | $S_H$, $S_F$ for fatigue and peak, with every ISO 6336 factor | `03_rating.rate_helical_pair` |
| Components | joint of each gear, shaft minimum diameters, bearing loads and selection, housing tolerances | `04_components.size_pair_components` |
| Manufacturing | tolerances, thickness allowances, backlash, $W_k$, $M_d$, data tables, DXF | `05_manufacturing` |

## Glossary

| Symbol | Name | Units | Notebook |
|:--:|:--|:--:|:--|
| $\beta$ | helix angle on the reference cylinder | ° | 01 |
| $\beta_b$ | base helix angle, $\sin\beta_b = \sin\beta\cos\alpha_n$ | ° | 01 |
| $\beta_w$ | operating helix angle on the pitch cylinder | ° | 01 |
| $m_n$, $m_t$ | normal and transverse module | mm | 01 |
| $\alpha_n$, $\alpha_t$, $\alpha_{wt}$ | normal, transverse and operating transverse pressure angle | ° | 01 |
| $p_x$, $p_z$ | axial pitch and lead | mm | 01 |
| $z_n$ | virtual tooth count | – | 01, 03 |
| $\varepsilon_\alpha$, $\varepsilon_\beta$, $\varepsilon_\gamma$ | transverse, overlap and total contact ratio | – | 01 |
| $x_n$ | profile shift coefficient in normal modules | – | 02 |
| $F_t$, $F_r$, $F_a$ | tangential, radial and axial tooth force | N | 02 |
| $Z_\beta$, $Y_\beta$ | helix angle factors for contact and bending | – | 03 |
| $W_k$ | span over $k$ teeth in the normal plane | mm | 01, 05 |
| $M_d$ | dimension over two balls | mm | 01, 05 |